# Charge Categorization QA

There's no ground-truth label for "correct category" here, so accuracy can't be checked against an answer key. Instead this notebook checks the classifier from `charge_categorization.ipynb` the way you'd audit a rule-based system:

1. **Coverage** — how many rows were classified by rule vs. TF-IDF fallback vs. not at all.
2. **Rule audit** — for every regex pattern in the taxonomy, which raw labels did it actually catch? This is where a too-generic pattern (e.g. matching `freight` inside something that isn't freight) would show up.
3. **Low-confidence fallback review** — the TF-IDF matches, sorted by similarity score so the weakest ones surface first.
4. **Consistency check** — does the same `Charge Type` ever get assigned to more than one Major Category (depending on its `Charge Description`)? That's a signal worth a manual look.
5. **Uncategorized bucket** — the full list of what's still in "Other / Uncategorized", not just the top 30.
6. **Random spot-check sample** — a reproducible random sample per major category to eyeball.

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

mapping = pd.read_csv("../outputs/charge_category_mapping.csv")
print(f"{len(mapping):,} unique (Charge Type, Charge Description) combinations, {mapping['row_count'].sum():,} rows total")
mapping.head()

3,032 unique (Charge Type, Charge Description) combinations, 256,764 rows total


,Charge Type,Charge Description,row_count,total_value,Major Category,Subcategory,Method
0,(CL175) CLASS 175.0 (EX175) 393 EX175 66803,(CL175) CLASS 175.0 (EX175) 393 EX175 66803,1,2625.36,Other / Uncategorized,Unclassified,none
1,(EX175) 103 EX175 84977,(EX175) 103 EX175 84977,1,875.26,Other / Uncategorized,Unclassified,none
2,(EX175) 117 EX175 42389,(EX175) 117 EX175 42389,1,495.95,Other / Uncategorized,Unclassified,none
3,(EX175) 123 EX175 91831,(EX175) 123 EX175 91831,1,1129.52,Other / Uncategorized,Unclassified,none
4,(EX175) 129 EX175 27658,(EX175) 129 EX175 27658,1,356.79,Other / Uncategorized,Unclassified,none


## Taxonomy (imported from `taxonomy.py`, shared with `charge_categorization.ipynb`)

Both notebooks import the same `TAXONOMY` / `PRIORITY_OVERRIDES` from `taxonomy.py`, so this notebook can re-derive *why* each combo was classified the way it was without risking the two copies drifting out of sync (they used to be pasted into each notebook separately).

In [2]:
import sys
sys.path.append("../src")
from taxonomy import TAXONOMY, CATCH_ALL, PRIORITY_OVERRIDES, TFIDF_EXCLUDE, normalize, rule_classify_row

## 1. Coverage: rule vs. TF-IDF fallback vs. none

In [3]:
coverage = mapping.groupby("Method").agg(combos=("Method", "size"), rows=("row_count", "sum"))
coverage["pct_of_rows"] = (coverage["rows"] / coverage["rows"].sum() * 100).round(1)
coverage.sort_values("rows", ascending=False)

,combos,rows,pct_of_rows
Method,,,
rule,1499,247136,96.3
tfidf_fallback,524,5593,2.2
none,1009,4035,1.6


## 2. Rule audit — which pattern caught what

Patterns match against `Charge Type` + `Charge Description` **combined into one string** (not Charge Type checked to completion first, then Charge Description as a fallback — that earlier version badly undercounted specific subcategories like "Ground" because a generic Charge Type like "Freight" matched the catch-all pattern before Charge Description was ever examined). For every combo classified by a rule, find the *exact* pattern that matched. Grouping by pattern shows you, per regex, how many rows it swept up and a few example raw labels — the fastest way to spot an over-broad pattern (e.g. a generic `\bbase\b` catching something that isn't a base freight charge).

In [4]:
def rule_match_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None
    for pat, result in PRIORITY_OVERRIDES:
        if re.search(pat, t):
            return result + ("[override] " + pat,)
    for major, subcats in TAXONOMY.items():
        for sub, patterns in subcats.items():
            for pat in patterns:
                if re.search(pat, t):
                    return (major, sub, pat)
    return None


rule_rows = mapping[mapping["Method"] == "rule"].copy()
detail = rule_rows.apply(lambda r: rule_match_row(r["Charge Type"], r["Charge Description"]), axis=1)
rule_rows["Matched Pattern"] = [d[2] if d else None for d in detail]

pattern_summary = (
    rule_rows.groupby(["Major Category", "Subcategory", "Matched Pattern"])
    .agg(
        combos=("Matched Pattern", "size"),
        rows=("row_count", "sum"),
        examples=("Charge Type", lambda s: ", ".join(sorted(set(s.dropna().astype(str)))[:4])),
    )
    .sort_values("rows", ascending=False)
)
pattern_summary

combos   rows                                                     examples
Major Category                   Subcategory                              Matched Pattern                                                                                                  
Fuel Surcharge                   Domestic Fuel Surcharge                  \bfuel surcharge\b                         18  63957  32.8% FUEL SURCHARGE, 33.1% FUEL SURCHARGE, 33.9% FUEL S...
Line Haul / Base Transportation  General / Mode Not Specified             transportation charge                       3  38402       Freight, Transportation Charge, Transportation Charges
Discounts                        General Discount                         \bdiscount\b                               84  36253  --> R&L'S DISCOUNT SAVES YOU, 75.00% DISCOUNT, 795 LESS ...
                                 Earned Discount                          earned discount                             1  22904                                              Earned Discount
Line Haul / Base Transportation  Ground                                   [override] \bground\b.*\bcommercial\b       7  16844                                                      Freight
...                                                                                                                 ...    ...                                                          ...
Taxes & Customs                  VAT                                      \bbtw\b                                     4      6  BTW, Btw, Toeslag fact. IR/BTW aan verzender, Totaal exc...
Line Haul / Base Transportation  Ground                                   \bltl\b                                     2      2  TIME CRITICAL GTD LTL STD, TIME CRITICAL GTD LTL STDÃƒÂ‚...
Accessorial / Delivery Surcharge Commercial / Business Delivery Surcharge \bcommercial\b                              1      2                                                  Accessorial
Administrative & Service Fees    Package Handling / Storage               warehouse storage                           1      1                                            Warehouse Storage
                                 Document Fee                             document fee                                1      1                                                 DOCUMENT FEE

[83 rows x 3 columns]

### The riskiest patterns to check by hand

The generic, single-word patterns (`\bfreight\b`, `\bbase\b`, `\bdiscount\b`, `\btax(es)?\b`, etc.) are the ones most likely to catch something that doesn't belong. Pull every distinct raw label each one matched so you can eyeball them directly.

In [5]:
GENERIC_PATTERNS = [r"\bfreight\b", r"\bbase\b", r"\bdiscount\b", r"\btax(es)?\b", r"\bstorage\b", r"\bpeak\b", r"\btoll\b", r"\bcommercial\b"]

for pat in GENERIC_PATTERNS:
    sub = rule_rows[rule_rows["Matched Pattern"] == pat]
    if sub.empty:
        continue
    print(f"--- pattern `{pat}` -> {sub['row_count'].sum():,} rows, {len(sub)} combos ---")
    print(sub[["Charge Type", "Charge Description", "row_count"]].head(15).to_string(index=False))
    print()

--- pattern `\bfreight\b` -> 1,828 rows, 802 combos ---
                      Charge Type                Charge Description  row_count
                    ALR FREIGHT 7                     ALR FREIGHT 7          1
                          FREIGHT                           FREIGHT          5
                   FREIGHT CHARGE                    FREIGHT CHARGE         32
FRT FREIGHT CHARGE 1.000 110.0000 FRT FREIGHT CHARGE 1.000 110.0000          2
FRT FREIGHT CHARGE 1.000 119.0000 FRT FREIGHT CHARGE 1.000 119.0000          1
FRT FREIGHT CHARGE 1.000 145.0000 FRT FREIGHT CHARGE 1.000 145.0000          1
FRT FREIGHT CHARGE 1.000 153.0000 FRT FREIGHT CHARGE 1.000 153.0000          1
 FRT FREIGHT CHARGE 1.000 60.0000  FRT FREIGHT CHARGE 1.000 60.0000          1
 FRT FREIGHT CHARGE 1.000 90.0000  FRT FREIGHT CHARGE 1.000 90.0000          1
     FRT FREIGHT CHARGE 1115.0000      FRT FREIGHT CHARGE 1115.0000          1
     FRT FREIGHT CHARGE 1155.0000      FRT FREIGHT CHARGE 1155.0000        

## 3. TF-IDF fallback review

These combos didn't hit any keyword rule — they were assigned by nearest-neighbor similarity to a subcategory's reference keywords. **Do not trust the similarity score alone**: a systematic check found matches sharing just one generic word with a reference doc (e.g. "International Processing Fee" vs. the "International Fuel Surcharge" reference text) can score 0.4–0.7, comfortably above the 0.25 acceptance threshold, while being flatly wrong. So this section reviews fallback matches two ways: sorted by *similarity* (weakest first) and sorted by *row impact* (biggest dollar/row exposure first) — a wrong match sitting at 0.6 similarity but touching 5,000 rows is a bigger problem than a wrong match at 0.26 similarity touching 3 rows, and the similarity-only sort would never surface it.

In [6]:
ref_docs, ref_labels = [], []
for major, subcats in TAXONOMY.items():
    for sub, patterns in subcats.items():
        if (major, sub) in TFIDF_EXCLUDE:
            continue
        doc = " ".join(p.replace(r"\b", "").replace(".*", " ").replace("'?", "").replace("?", "") for p in patterns)
        ref_docs.append(doc)
        ref_labels.append((major, sub))

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
ref_vectors = vectorizer.fit_transform(ref_docs)


def best_similarity_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None, None
    vec = vectorizer.transform([t])
    sims = cosine_similarity(vec, ref_vectors)[0]
    bi = sims.argmax()
    return ref_labels[bi], sims[bi]


fallback_rows = mapping[mapping["Method"] == "tfidf_fallback"].copy()

sims = []
for _, r in fallback_rows.iterrows():
    _, score = best_similarity_row(r["Charge Type"], r["Charge Description"])
    sims.append(score)

fallback_rows["similarity"] = sims

print(f"{len(fallback_rows):,} combos classified via TF-IDF fallback, {fallback_rows['row_count'].sum():,} rows")
print()
print("--- weakest matches (lowest similarity first) ---")
display(fallback_rows.sort_values("similarity")[["Charge Type", "Charge Description", "Major Category", "Subcategory", "similarity", "row_count"]].head(20))
print()
print("--- highest-impact matches (most rows first) -- review these even though confidence may look high ---")
display(fallback_rows.sort_values("row_count", ascending=False)[["Charge Type", "Charge Description", "Major Category", "Subcategory", "similarity", "row_count"]].head(20))

524 combos classified via TF-IDF fallback, 5,593 rows

--- weakest matches (lowest similarity first) ---


,Charge Type,Charge Description,Major Category,Subcategory,similarity,row_count
2997,WW Expedited,WW Expedited,Line Haul / Base Transportation,International / Export / Import Freight,0.251650,21
389,Accessorial,Detention,Accessorial / Delivery Surcharge,Residential Delivery/Surcharge,0.252224,1
2523,PARK PARKING FEES @ PU,PARK PARKING FEES @ PU,Administrative & Service Fees,Third Party Billing,0.252801,1
2522,PARK PARKING FEES,PARK PARKING FEES,Administrative & Service Fees,Third Party Billing,0.252801,15
2086,Kosten service klantoplossing,Kosten service klantoplossing,Taxes & Customs,Customs / Brokerage,0.252952,2
2899,TRUCKING FEE,TRUCKING FEE,Line Haul / Base Transportation,Ground,0.253759,2
2869,TERMINAL CHARGE,TERMINAL CHARGE,Accessorial / Delivery Surcharge,Demand / Peak Surcharge,0.254658,6
737,Courier service DHL 0,Courier service DHL 0,Credits,Credit From Carrier,0.255486,1
822,DOCU LESS THAN 2 HOUR NOTICE,DOCU LESS THAN 2 HOUR NOTICE,Administrative & Service Fees,Document Fee,0.255498,3
254,Accessorial,CBI INSPECTION OF SHIPMENT FEE,Administrative & Service Fees,Billing Adjustment / Correction,0.255534,11



--- highest-impact matches (most rows first) -- review these even though confidence may look high ---


,Charge Type,Charge Description,Major Category,Subcategory,similarity,row_count
2864,TB Express Saver,TB Express Saver,Line Haul / Base Transportation,International / Export / Import Freight,0.580888,1556
2530,PGA Processing Fee,PGA Processing Fee,Taxes & Customs,Customs / Brokerage,0.262092,185
2126,Linehaul,Linehaul,Line Haul / Base Transportation,General / Mode Not Specified,0.307118,181
2935,US Inbound Processing Fee,US Inbound Processing Fee,Taxes & Customs,Customs / Brokerage,0.258449,159
2527,PGA Disclaim Fee,PGA Disclaim Fee,Discounts,General Discount,0.331032,118
790,DELIVERY,DELIVERY,Accessorial / Delivery Surcharge,Delivery Area Surcharge (DAS),0.656485,103
827,DOCUMENT MANAGEMENT,DOCUMENT MANAGEMENT,Administrative & Service Fees,Document Fee,0.690418,95
1028,Express,Express,Line Haul / Base Transportation,International / Export / Import Freight,0.622713,93
2528,PGA FILING,PGA FILING,Accessorial / Delivery Surcharge,Additional Handling,0.335128,85
3012,Weekend Charge,Weekend Charge,Accessorial / Delivery Surcharge,Demand / Peak Surcharge,0.286584,85


## 4. Consistency check — does the same Charge Type ever land in more than one Major Category?

Most Charge Types should map to exactly one category. If a Charge Type spans multiple categories depending on its Charge Description, that's worth a manual look — either it's a genuinely mixed bucket (carrier reuses one Charge Type label for several kinds of fees) or a pattern is over-firing on the Description field.

In [7]:
type_consistency = (
    mapping.groupby("Charge Type")["Major Category"]
    .agg(lambda s: sorted(set(s)))
    .reset_index()
)
type_consistency["n_categories"] = type_consistency["Major Category"].apply(len)
inconsistent = type_consistency[type_consistency["n_categories"] > 1].sort_values("n_categories", ascending=False)
print(f"{len(inconsistent)} Charge Type values span more than one Major Category")
inconsistent

7 Charge Type values span more than one Major Category


,Charge Type,Major Category,n_categories
238,Accessorial,"[Accessorial / Delivery Surcharge, Administrative & Serv...",6
240,Accessorial 2,"[Accessorial / Delivery Surcharge, Administrative & Serv...",5
241,Accessorial 3,"[Accessorial / Delivery Surcharge, Administrative & Serv...",5
239,Accessorial 1,"[Administrative & Service Fees, Fuel Surcharge, Other / ...",4
242,Accessorial 4,"[Administrative & Service Fees, Fuel Surcharge, Other / ...",4
760,Freight,"[Accessorial / Delivery Surcharge, Administrative & Serv...",4
243,Accessorial 5,"[Administrative & Service Fees, Taxes & Customs]",2


## 5. Full uncategorized list

Everything still in "Other / Uncategorized" — not caught by a rule or by the TF-IDF fallback threshold (0.25 similarity).

In [8]:
uncategorized = mapping[mapping["Major Category"] == CATCH_ALL[0]].sort_values("row_count", ascending=False).reset_index(drop=True)
print(f"{len(uncategorized):,} uncategorized combinations, {uncategorized['row_count'].sum():,} rows, ${uncategorized['total_value'].sum():,.0f} total value")
uncategorized.to_csv("../outputs/charge_uncategorized_full.csv", index=False)
print("Saved full list to ../outputs/charge_uncategorized_full.csv")
uncategorized.head(50)

1,009 uncategorized combinations, 4,035 rows, $634,599 total value
Saved full list to outputs/charge_uncategorized_full.csv


,Charge Type,Charge Description,row_count,total_value,Major Category,Subcategory,Method
0,Pps Critical (American Airlines),Pps Critical (American Airlines),308,26965.40,Other / Uncategorized,Unclassified,none
1,PREMIUM 12:00,PREMIUM 12:00,235,1175.00,Other / Uncategorized,Unclassified,none
2,Entry Prep Fee,Entry Prep Fee,122,5882.59,Other / Uncategorized,Unclassified,none
3,CONTRACT MINIMUM REACHED MCADJ ADJ,CONTRACT MINIMUM REACHED MCADJ ADJ,122,9482.55,Other / Uncategorized,Unclassified,none
4,Performance Pricing,Performance Pricing,118,-2522.82,Other / Uncategorized,Unclassified,none
5,Order Lane Charge,Order Lane Charge,116,51543.76,Other / Uncategorized,Unclassified,none
6,Service Charge,Service Charge,90,3469.00,Other / Uncategorized,Unclassified,none
7,Retourzendingen 3 UPS Ophaalpogingen,Retourzendingen 3 UPS Ophaalpogingen,67,794.20,Other / Uncategorized,Unclassified,none
8,Standard to Canada,Standard to Canada,63,2662.05,Other / Uncategorized,Unclassified,none
9,Accessorial,LIMITED ACCESS CHARGE FLAT,57,3067.74,Other / Uncategorized,Unclassified,none


## 6. Random spot-check sample

A reproducible random sample from each major category, for manually eyeballing whether the assignment makes sense.

In [9]:
SEED = 42
sample = (
    mapping.groupby("Major Category", group_keys=False)
    .apply(lambda g: g.sample(n=min(5, len(g)), random_state=SEED))
    .sort_values(["Major Category", "row_count"], ascending=[True, False])
)
sample[["Major Category", "Subcategory", "Charge Type", "Charge Description", "Method", "row_count"]]

,Major Category,Subcategory,Charge Type,Charge Description,Method,row_count
515,Accessorial / Delivery Surcharge,Additional Handling,Additional Handling - Length+Girth,Additional Handling - Length+Girth,rule,43
2496,Accessorial / Delivery Surcharge,Additional Handling,Origin Terminal Handling Charge (THC),Origin Terminal Handling Charge (THC),tfidf_fallback,2
412,Accessorial / Delivery Surcharge,Additional Handling,Accessorial,LIFTGATE REQUIRED @P/U FEE,rule,2
2494,Accessorial / Delivery Surcharge,Additional Handling,Origin Handling Fee Busan port charge,Origin Handling Fee Busan port charge,tfidf_fallback,1
394,Accessorial / Delivery Surcharge,Delivery Area Surcharge (DAS),Accessorial,FLAT NWCARBEA SURCHARGE,tfidf_fallback,1
2660,Administrative & Service Fees,Billing Adjustment / Correction,Rebill Fee,Rebill Fee,rule,47
175,Administrative & Service Fees,Document Fee,ADMI N DOCS E,ADMI N DOCS E,tfidf_fallback,7
2817,Administrative & Service Fees,Billing Adjustment / Correction,Shipping Charge Correction Large Package Surcharge - Len...,Shipping Charge Correction Large Package Surcharge - Len...,rule,6
2169,Administrative & Service Fees,Third Party Billing,MISCELLANEOUS BILLING,MISCELLANEOUS BILLING,tfidf_fallback,3
610,Administrative & Service Fees,Billing Adjustment / Correction,Billing Adjustment for W/E 07/18/2026,Billing Adjustment for W/E 07/18/2026,rule,2
